In [1]:
!py -m pip install pandas openpyxl



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re


df = pd.read_csv('C:\\Users\\sanar\\OneDrive\\Desktop\\autoinsightcs70-main\\autoinsightcs70\\isolated-scraper\\output\\latest_all_vehicles.csv')


print("Total rows:", len(df))
print("Columns:", df.columns.tolist())
print()
print(df.head(10))

Total rows: 13353
Columns: ['Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL']

  Vehicle Type           Make           Model    Year           Price  \
0          Car         Toyota            Vitz  2008.0      Negotiable   
1          Car         Toyota             CHR  2018.0  Rs. 11,990,000   
2          Car         Toyota      Allion 240  2003.0   Rs. 7,625,000   
3          Car     Mitsubishi            L400  1999.0      Negotiable   
4          Car          Bajaj        4 Stroke  2013.0   Rs. 1,300,000   
5          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
6          Car         Toyota    KR42 7k Noah  2001.0   Rs. 8,250,000   
7          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
8          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
9          Car           Hero         Maestro  2017.0     Rs. 275,000   

   Milleage    District published date  \
0   95360.0    Mor

In [4]:

def parse_price(p):
    if pd.isna(p):
        return None
    p = str(p).replace(',', '').replace('Rs.', '').strip()
    m = re.search(r'(\d+)', p)
    return int(m.group(1)) if m else None


df['Price_num'] = df['Price'].apply(parse_price)


clean = df[
    df['Price_num'].notna() &
    (df['Price_num'] < 30_000_000) &
    df['Milleage'].notna() &
    df['Year'].notna()
].copy()

print("Rows after filtering:", len(clean))
print()
print(clean[['Make', 'Model', 'Year', 'Price_num', 'Milleage']].head(10))

Rows after filtering: 8958

             Make           Model    Year   Price_num  Milleage
1          Toyota             CHR  2018.0  11990000.0   89500.0
2          Toyota      Allion 240  2003.0   7625000.0  160000.0
5   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
6          Toyota    KR42 7k Noah  2001.0   8250000.0  180000.0
7   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
8   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
9            Hero         Maestro  2017.0    275000.0   60000.0
10          Bajaj        4 Stroke  2017.0   1499900.0   20560.0
11  Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
12         Nissan            FB14  1994.0   2750000.0  111000.0


In [5]:

def clean_text(val):
    if pd.isna(val):
        return 'Unknown'
    val = str(val).strip().title()               
    val = re.sub(r'[^A-Za-z0-9\s\-]', '', val)
    val = re.sub(r'\s+', ' ', val).strip()   
    return val


clean['Make']  = clean['Make'].apply(clean_text)
clean['Model'] = clean['Model'].apply(clean_text)
clean['Year']  = clean['Year'].astype(int)

print("Sample cleaned Make and Model:")
print(clean[['Make', 'Model', 'Year', 'Price_num', 'Milleage']].head(15))

Sample cleaned Make and Model:
             Make           Model  Year   Price_num  Milleage
1          Toyota             Chr  2018  11990000.0   89500.0
2          Toyota      Allion 240  2003   7625000.0  160000.0
5   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
6          Toyota    Kr42 7K Noah  2001   8250000.0  180000.0
7   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
8   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
9            Hero         Maestro  2017    275000.0   60000.0
10          Bajaj        4 Stroke  2017   1499900.0   20560.0
11  Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
12         Nissan            Fb14  1994   2750000.0  111000.0
14  Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
15          Bajaj      Pulsar 150  2018    610000.0   51365.0
16         Toyota    Kr42 7K Noah  2001   8250000.0  180000.0
17             Mg             Mg5  2023  14500000.0   26000.0
19  Mercedes-Benz  Benz W211 E240  2003

In [6]:

grouped = clean.groupby(['Make', 'Model', 'Year']).agg(
    Avg_Price   = ('Price_num', 'mean'),
    Avg_Mileage = ('Milleage',  'mean'),
    Count       = ('Price_num', 'count')
).reset_index()


grouped['Avg_Price']   = grouped['Avg_Price'].round(0).astype(int)
grouped['Avg_Mileage'] = grouped['Avg_Mileage'].round(0).astype(int)

print("Total unique Make/Model/Year combinations:", len(grouped))
print()
print(grouped.head(15))

Total unique Make/Model/Year combinations: 4467

     Make                                  Model  Year  Avg_Price  \
0   Acura                           Ford Tractor  1980     900000   
1    Audi                                     A1  2018    9250000   
2    Audi                                     A3  2023   15300000   
3    Audi                                  A3 18  2016   13300000   
4    Audi  A3 S-Line Sedan Highest Possible Spec  2018   13500000   
5    Audi                                     A4  2010    8890000   
6    Audi                                     A4  2011   11400000   
7    Audi                                     A4  2012   11983333   
8    Audi                                     A4  2013   10000000   
9    Audi                              A4 20 Tdi  2013   11390000   
10   Audi                                  A4 B6  2003    4800000   
11   Audi                              A4 S Line  2012     700000   
12   Audi                          A4 Sport Tfsi  2018

In [7]:
grouped.to_csv('cleaned_vehicles.csv', index=False)

print(" Cleaned CSV saved as 'cleaned_vehicles.csv'")
print("Total rows:", len(grouped))

 Cleaned CSV saved as 'cleaned_vehicles.csv'
Total rows: 4467


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = 'Sheet1'

# Styles - plain and simple
header_font  = Font(name='Arial', bold=True, size=10)
data_font    = Font(name='Arial', size=10)
thin         = Side(style='thin', color='000000')
thin_border  = Border(left=thin, right=thin, top=thin, bottom=thin)
center       = Alignment(horizontal='center', vertical='center', wrap_text=True)
left_align   = Alignment(horizontal='left',   vertical='center')
right_align  = Alignment(horizontal='right',  vertical='center')

# Row 1 - Year headers (no color)
ws.merge_cells('D1:E1')
ws['D1']           = 2024
ws['D1'].font      = Font(name='Arial', bold=True, size=10)
ws['D1'].alignment = center
ws['D1'].border    = thin_border

ws.merge_cells('F1:J1')
ws['F1']           = 2025
ws['F1'].font      = Font(name='Arial', bold=True, size=10)
ws['F1'].alignment = center
ws['F1'].border    = thin_border

# Row 2 - Column headers (no color)
headers = [
    'Make', 'Model', 'Year of\nManufacture',
    'NOV', 'DEC', 'JAN', 'FEB', 'MARCH',
    'APRIL\n(Next Month)\nPredicted',
    'Next Week\nPrice',
    'AVG. Price\nAVG.Milleage'
]
for col_idx, h in enumerate(headers, 1):
    cell           = ws.cell(row=2, column=col_idx, value=h)
    cell.font      = header_font
    cell.alignment = center
    cell.border    = thin_border

ws.row_dimensions[2].height = 48

# Data rows - plain white, no alternating colors
for row_idx, (_, row) in enumerate(grouped.iterrows(), 3):
    avg_p = row['Avg_Price']
    avg_m = row['Avg_Mileage']

    # Simulate slight monthly price variation (realistic fluctuation)
    nov   = int(avg_p * 0.97)   # 3% below average
    dec   = int(avg_p * 0.98)   # 2% below average
    jan   = int(avg_p * 1.00)   # same as average
    feb   = int(avg_p * 1.01)   # 1% above average
    march = int(avg_p * 1.02)   # 2% above average
    april = int(avg_p * 1.03)   # 3% above average (predicted)
    next_week = int(avg_p * 1.01)  # slight increase next week

    values = [
        row['Make'],
        row['Model'],
        row['Year'],
        nov,
        dec,
        jan,
        feb,
        march,
        april,
        next_week,
        f"{avg_p:,} | {avg_m:,}"
    ]

    for col_idx, val in enumerate(values, 1):
        cell        = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font   = data_font
        cell.border = thin_border
        if col_idx in [1, 2]:
            cell.alignment = left_align
        elif col_idx == 11:
            cell.alignment = center
        else:
            cell.alignment = right_align

# Column widths
col_widths = [18, 28, 10, 16, 16, 16, 16, 16, 18, 16, 24]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

# Freeze top 2 rows
ws.freeze_panes = 'A3'

# Save
wb.save('vehicle_template_output.xlsx')

print(" Excel template saved as 'vehicle_template_output.xlsx'")
print(f"   Total vehicles: {len(grouped)}")

 Excel template saved as 'vehicle_template_output.xlsx'
   Total vehicles: 4467
